NER CON BERT FINE-TUNED: ESTRAZIONE AVANZATA DI ENTITA'

NER Named entity recognizion

Nel NER non vogliamo classificare l'intera frase (esempio se una recensione è positiva/negativa), vogliamo classificare ogni token della frase.
Esempio: "Apple ha aperto una sede a Milano"
Apple - ORG
ha - O
aperto - O
una - O
sede - O
a - O
Milano - LOC
(O=token che non appartiene a nessuna entità)

Mentre nel sentiment analysis è importante 'valutare' l'intera frase, nel NER sono importanti ogni singola parola; 
mentre nel sentiment analysis prendiamo solo CLS (come se fosse una foto di gruppo dell'intera frase), nel NER prendiamo tutto last_hidden_state, cioè la rappresentazione di tutti i token (è come identificare ogni singolo elemento nella foto)

Se BERT ha tokenizzato la precedente frase come: [CLS] - Apple - ha - aperto - una - sede - a - Milano- [SEP]
BERT produce, per ogni token 768 valori
Quindi last_hidden_state.shape=[batch_size, sequence_lenght,768], se abbiamo una sola frase di 9 token [1,9,768]
A questo punt, sul last_hidde_state, aggiungiamo una classification head che lavora su ogni token. Il classificatore, per ogni token, prende i 768 valori e produce una percentuale di confidenze per ogni etichetta che vogliamo produrre.

Per ogni token i, il modello calcola un vettore di punteggi grezzi che viene normalizzato per ottenre la probabilità della classe k. viene calcolata una % per ogni classe
Questo processo avviene in parallelo per tutti i token della sequenza, rendendo l'inferenza estremamente veloce rispetto alle RNN

Questa è la grande differenza tra il sentiment analysis e NER
Sentiment - 1 frase - 1 classificazione
NER - 1 frase - una classificazione per ogni token

Se invece analizziamo la frase "Leonardo Da Vinci naque in Italia", il sistema BIO è in grado di capire che "Leonardo Da Vinci" fa parte della stessa entità. Per farlo utilizza 3 lettere:
B = Beginning
    inizio entità
I = Inside
    continuazione entità
O = Outside
    token che non appartiene a nessuna entità
Quindi la frase "Leonardo Da Vinci naque in Italia" diventa:
Leonardo    B-PER
Da          I-PER
Vinci       I-PER
naque       O
in          O
Italia      B-LOC
Questo perchè Leonardo Da Vinci è un'unica persona (PER)
Se il modello dicesse soltanto: Leonardo PER, Da PER, Vinci PER non sarebbe chiaro se fossero 3 entità separate oppure una sola.

Adesso arriviamo al fine-tunning

Prendi BERT già preaddestrato, BERT conosce il linguaggio ma non neccessariemente le tue etichette NER
A questo punto gli dai un dataset etichettato (con le tue etichette), la rete diventa:
BERT pre-addestrato -> classificatio head -> etihcette NER personalizzate

Durante il training:
testo - BERT - predizione per goni token - confronto con etichetta corretta - loss - backpropagation - aggiornamento dei pesi.

Con Huggin Face trovi classi del tipo:
AutoModelForSequenceClassification, per classificare l'intero testo, utile per sentiment analysis
e
AutoModelForTokenClassification, per NER in cui devi classificare ogni token

Non devi costruire manualmente tutot BERT, Huggin Face ti da:
tokenizer + BERT + classification head NER + eventuali pesi già fine-tuned

E' importanto distinguere due casi:
1) lavori con classificazioni standard: in questo caso qualcun altro ha già fatto fine-tuning e tu fai solo inferenza. Quindi scarichi BERT già fine-tuned oper NER e lo usi direttamente
2) lavori con una classificazione personalizzata: in questo caso non vuoi classificare in modo standard, ma con tue etichette. Quindi prendi BERT generico, gli dai il tuo dataset NER, e fai fine-tuning. In questo modo specializzi BERT.

Attenzione: BERT non tokenizza necessariamente parola per parola
quindi possiamo avere parole che corrispondono ad un solo token
e parole, meno comuni, che potrebbero essere spezzate in più token
Esempio:
parola: "Microsoft"   token:"Microsoft"
parola: "qualcosalunga" token: "qual" "##cosa" "##lunga"
questa è la logica di WordPiece

A questo punto devi allenare le etichette originale al subtoken
Questa è una delle parti più delicate del training NER con Transformer.
Le librerie di Hugging Face forniscono strumenti che aiutano a mantenere il collegamento tra token originali e subtoken, ma è da ricordare che
nella NER con BERT le etichette devono essere correttamente allineato alla tokenizzazione del modello

BERT ha miglioarato i task di sequence labeling rispetto ai sistemi precedenti basati solo su rappresentazioni statiche.
BERT guarda il token + parole prima + parole dopo e crea una rappresentazione contestuale NER
Questo permette di distinguere tra:
"Ho mangiato una apple" e "Apple ha aperto un nuovo negozio"
Questo grazie ai transformer

Nel NER con BERT fine-tuned, BERT produce una rappresentazione contestuale per ogni token e una testa di classificazione assegna a ciascun token un’etichetta come PER, ORG, LOC o O; il fine-tuning adatta i pesi del modello al dataset NER specifico.

Analisi delle Performance
- Metriche F1-score: mentre spacY eccelle nella velocità, i modelli basati su BERT mostrano spesso un incremendo dell F1-score superiore al 10% su dastaset complessi
- Costo Computazionale: il miglioramento dell'accuratezza richiede l'uso di GPU per l'inferenza, a differenza dei modelli statistici leggeri che girano velocemente su CPU. BERT è pesante, il compromesso è tra la velcoità pura dei modelli statistici e la comprensione profonda di BERT
- Fine-tunning mirato: BERT può essere adattato a linghe specifiche o gerghi tecnici con molti meno dati rispetto a un modello addestrato da zero.

BERT Specializzazione del Dominio
Invece di utilizzare BERT base, allenato su wikipeida, abbiamo a disposizoine domini specifici di BERT: varianti come BioBERT o SciBERT hanno visto entità tecniche durante il pre-training. Sono come medici che hanno studiato su manuali giusti, conoscono molecole o patologie rare come fossero termini comuni.
Oppure possiamo sfruttare la capacità di Zero-Shot NER cioè la capacità dei modelli più recenti di identificare tipi di entità nuovi (categorie mai viste prima) semplicemente fornendo solo una descrizione testuale.

Ottimizzazione della Pipeline
A volte BERT può sbagliare la coerenza, potrebbe mettere un tag I (insede) senza il B (begin) precedente, per evitare questo spesso aggiungiamo un lauyer CRF finale che lavora sopra BERT, agisce come un correttore grammaticale dei TAG.
Inoltre dobbiamo cambiare il modo in cui valutiamo il modello
Non ci interessa se ha indivinato l'80% dei token, ci interessa se ha identificato l'intero intervallo, un nome indovinato a metà è un errore 100%

In [5]:
"""
====================================================================================================
SFIDA NER: BIO-BERT (TRANSFORMERS) VS SPACY (MODELLI STATISTICI)
====================================================================================================
In questo script mettiamo a confronto due generazioni di Intelligenza Artificiale:
1. spaCy: Rappresenta l'approccio classico statistico (veloce, ma limitato dal contesto locale).
2. BioBERT: Rappresenta lo stato dell'arte dei Transformers (lento, ma con una comprensione 
   profonda e semantica del dominio medico).

Obiettivo: Dimostrare come BERT riesca a "leggere tra le righe" nei settori specialistici.
====================================================================================================
"""

import os

# --- CONFIGURAZIONE BACKEND ---
# Keras 3 è agnostico: qui forziamo l'uso di PyTorch come "motore" di calcolo.
# Questa riga deve essere eseguita PRIMA di importare keras o altri framework.
os.environ["KERAS_BACKEND"] = "torch"

import spacy
from transformers import pipeline

def load_models():
    """
    Inizializza e carica in memoria i due cervelli artificiali che confronteremo.
    
    Interazione:
    - nlp_spacy: Carica pesi statistici pre-calcolati per la lingua inglese generica.
    - ner_biobert: Scarica (se non presente) un modello BERT mastodontico addestrato su PubMed.
    """
    print("\n[INFO] Caricamento 'cervelli' in corso...")
    
    # --- SFIDANTE 1: spaCy (Tradizione) ---
    try:
        # Carica il modello statistico "large" per l'inglese.
        # È basato su una pipeline di regole e pesi lineari molto veloci.
        nlp_spacy = spacy.load("en_core_web_lg")
        print("[OK] spaCy caricato (Modello Statistico).")
    except:
        # Se il modello non è installato (es. manca il download), gestiamo l'errore.
        print("[!] Attenzione: Modello spaCy 'en_core_web_sm' non trovato.")
        nlp_spacy = None

    # --- SFIDANTE 2: BioBERT (Innovazione) ---
    # Il nome del modello su HuggingFace: una versione di BERT specializzata in medicina.
    model_name = "d4data/biomedical-ner-all"
    
    # La 'pipeline' è l'astrazione più alta possibile:
    # 1. Carica il Tokenizer (per spezzare le parole in numeri).
    # 2. Carica il Modello (i miliardi di parametri di BERT).
    # 3. Imposta l'aggregation_strategy="simple" per unire i pezzi di parole (##) in entità intere.
    # 4. framework="pt": Forza l'uso di PyTorch (fondamentale con Keras 3 installato).
    ner_biomedical = pipeline("ner",model=model_name,aggregation_strategy="average",device="cpu")
    print("[OK] BioBERT caricato (Modello Transformer di settore).")
    
    return nlp_spacy, ner_biomedical

def run_comparison(text):
    """
    Funzione principale che mette i due modelli davanti allo stesso testo specialistico.
    
    Parametri:
    - text: La frase medica complessa da analizzare.
    """
    # 1. Otteniamo i modelli pronti all'uso
    nlp_spacy, ner_biobert = load_models()
    
    print(f"\n{'='*70}")
    print(f"TESTO DA ANALIZZARE:\n'{text}'")
    print(f"{'='*70}")

    # --- FASE 1: ANALISI CON SPACY ---
    print("\n>>> ANALISI CON SPACY (Modello Statistico Generalista):")
    if nlp_spacy:
        # spaCy processa il testo in un colpo solo creando un oggetto 'Doc'
        doc = nlp_spacy(text)
        
        # doc.ents contiene le entità che spaCy ha 'indovinato'
        if not doc.ents:
            print("  [X] Nessuna entità medica trovata. spaCy non riconosce termini tecnici.")
        for ent in doc.ents:
            # ent.text: la parola trovata | ent.label_: la categoria (es. PERSON, ORG)
            print(f"  - Trovato: {ent.text:25} | Categoria: {ent.label_}")
    else:
        print("  [!] spaCy non disponibile.")

    # --- FASE 2: ANALISI CON BIO-BERT ---
    print("\n>>> ANALISI CON BIO-BERT (Deep Learning di Settore):")
    
    # La pipeline chiamerà internamente BERT, calcolerà l'attenzione e restituirà una lista
    results = ner_biobert(text)
    
    if not results:
        print("  [X] Nessuna entità trovata.")
    for ent in results:
        # ent['word']: il termine medico identificato
        # ent['entity_group']: la classe (es. DISEASE, DRUG)
        # ent['score']: quanto BERT è sicuro della sua risposta (da 0 a 1)
        print(f"  - Trovato: {ent['word']:25} | Categoria: {ent['entity_group']:8} | Confidenza: {ent['score']:.2f}")

# --- PUNTO DI INGRESSO (Main) ---
if __name__ == "__main__":
    # Scegliamo una frase "trappola": contiene termini medici che sembrano parole comuni.
    # 'Acute Myocardial Infarction' (Infarto) è un'entità complessa e annidata.
    clinical_case = (
        "The patient showed symptoms of Acute Myocardial Infarction "
        "and was treated with Aspirin and Heparin to inhibit Platelet aggregation."
    )
    
    # Avviamo il confronto
    run_comparison(clinical_case)

# --- GUIDA ALLA LETTURA DEI RISULTATI PER STUDENTI ---
"""
PERCHÉ VEDI QUELLO CHE VEDI? (Analisi riga per riga dei concetti):

1. LA STRUTTURA DEL CODICE: Il codice è diviso in 'Caricamento' (pesante) ed 'Esecuzione' (veloce).
   In AI, caricare i modelli in memoria è l'operazione più costosa; una volta pronti, 
   possono analizzare migliaia di frasi.

2. IL "COMPORTAMENTO" DI SPACY: Noterai che spaCy potrebbe scansionare 'Aspirin' ma ignorare 
   'Platelet aggregation'. Questo perché spaCy non "capisce" il senso, cerca solo pattern 
   statistici comuni. Se una parola non era nel suo dizionario di addestramento, non esiste.

3. IL "RAGIONAMENTO" DI BERT: BioBERT invece guarda la parola 'Platelet' e vede che è vicina 
   a 'inhibit'. Grazie al meccanismo di SELF-ATTENTION (Slide 8), capisce che 'aggregation' 
   è collegata a 'Platelet' e formano un unico concetto biologico.

4. AGGREGATION STRATEGY: La riga 'aggregation_strategy="simple"' è fondamentale. 
   Poiché BERT spezza le parole lunghe (es. 'Infarction' -> 'Infar', '##ction'), 
   questa opzione dice alla pipeline: "Riappiccica i pezzi prima di mostrarmeli".

5. LE CATEGORIE: Noterai che BioBERT usa classi come 'DISEASE' o 'DRUG', mentre spaCy usa 
   'ORG' o 'GPE'. Questa è la forza della 'Domain Specialization' (Slide 12).
"""


[INFO] Caricamento 'cervelli' in corso...
[OK] spaCy caricato (Modello Statistico).


Loading weights: 100%|██████████| 102/102 [00:00<00:00, 2544.10it/s]


[OK] BioBERT caricato (Modello Transformer di settore).

TESTO DA ANALIZZARE:
'The patient showed symptoms of Acute Myocardial Infarction and was treated with Aspirin and Heparin to inhibit Platelet aggregation.'

>>> ANALISI CON SPACY (Modello Statistico Generalista):
  [X] Nessuna entità medica trovata. spaCy non riconosce termini tecnici.

>>> ANALISI CON BIO-BERT (Deep Learning di Settore):
  - Trovato: acute                     | Categoria: Detailed_description | Confidenza: 1.00
  - Trovato: myocardial infarction     | Categoria: Disease_disorder | Confidenza: 0.83
  - Trovato: aspirin                   | Categoria: Medication | Confidenza: 0.92
  - Trovato: heparin                   | Categoria: Medication | Confidenza: 0.66


'\nPERCHÉ VEDI QUELLO CHE VEDI? (Analisi riga per riga dei concetti):\n\n1. LA STRUTTURA DEL CODICE: Il codice è diviso in \'Caricamento\' (pesante) ed \'Esecuzione\' (veloce).\n   In AI, caricare i modelli in memoria è l\'operazione più costosa; una volta pronti, \n   possono analizzare migliaia di frasi.\n\n2. IL "COMPORTAMENTO" DI SPACY: Noterai che spaCy potrebbe scansionare \'Aspirin\' ma ignorare \n   \'Platelet aggregation\'. Questo perché spaCy non "capisce" il senso, cerca solo pattern \n   statistici comuni. Se una parola non era nel suo dizionario di addestramento, non esiste.\n\n3. IL "RAGIONAMENTO" DI BERT: BioBERT invece guarda la parola \'Platelet\' e vede che è vicina \n   a \'inhibit\'. Grazie al meccanismo di SELF-ATTENTION (Slide 8), capisce che \'aggregation\' \n   è collegata a \'Platelet\' e formano un unico concetto biologico.\n\n4. AGGREGATION STRATEGY: La riga \'aggregation_strategy="simple"\' è fondamentale. \n   Poiché BERT spezza le parole lunghe (es. \'Infa